in this section i want and mask with orginal images and extract features from this images


part1: AND images with masks

In [1]:
from attention_unet import (
    AttentionUNet
)
import cv2
import os
import glob
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from torch.utils.data import Dataset

class CorneaDataset(Dataset):
    def __init__(self, images_list, masks_list, transform=None):
        self.images_list = images_list
        self.masks_list = masks_list
        self.transform = transform

    def __len__(self):
        return len(self.images_list)

    def __getitem__(self, idx):
        img_path = self.images_list[idx]
        mask_path = self.masks_list[idx]

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)

        mask = (mask > 0.5).float()

        return image, mask


load segmentation model

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AttentionUNet().to(device)
model.load_state_dict(torch.load(r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\segment\attention_unet_fold_2.pth", map_location=device))
model.eval()

images_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\segment\images"
labels_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\segment\corneaLabels"

images_list = sorted(glob.glob(os.path.join(images_dir, "*.*")))
masks_list = sorted(glob.glob(os.path.join(labels_dir, "*.*")))

kf = KFold(n_splits=5, shuffle=True, random_state=42)
splits = list(kf.split(images_list))
fold = 2 
train_idx, val_idx = splits[fold]

test_dataset = CorneaDataset(
    images_list=[images_list[i] for i in train_idx],
    masks_list=[masks_list[i] for i in train_idx],
    transform=transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor()
    ])
)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

AND segmented image with image and save

In [3]:
output_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\AND_result"
os.makedirs(output_dir, exist_ok=True)

with torch.no_grad():
    for idx, (images, masks) in enumerate(test_loader):
            
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        
        img_np = images.cpu().squeeze().permute(1, 2, 0).numpy()
        pred_np = torch.sigmoid(outputs).cpu().squeeze().numpy()
        
        pred_inv = 1 - (pred_np > 0.5).astype(np.float32)
        pred_inv_uint8 = (pred_inv * 255).astype(np.uint8)
        
        img_uint8 = (img_np * 255).astype(np.uint8)
        and_result = cv2.bitwise_and(img_uint8, img_uint8, mask=pred_inv_uint8)
        
        Image.fromarray(and_result).save(os.path.join(output_dir, f'and_result_{idx}.png'))



part2: pretrained model